# 0. Overview

The representative profile space is generated independently from multiple demographic datasets.

Although each attribute is statistically plausible, some attribute combinations are logically inconsistent.

This notebook applies a series of logical constraints to improve internal consistency before constructing behaviour profiles.

Input dataset:

- representative_profile_space.parquet

Output:

- constrained_representative_profile_space.parquet

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

### 2. Set Up File Paths

In [2]:
from pathlib import Path

project_root = Path.cwd().parents[1]

input_path = (
    project_root
    / "output"
    / "representative_profiles"
)

output_path = (
    project_root
    / "output"
    / "representative_profiles"
)

output_path.mkdir(
    parents=True,
    exist_ok=True
)

### 3. Load Dataset

In [3]:
profile_space_file = (
    input_path
    / "representative_profile_space.parquet"
)

profile_space = pd.read_parquet(
    profile_space_file
)

display(profile_space.head())

print(profile_space.shape)

,Age,Age Population,Sex,Sex Population,Household Category,Household Count,Economic Activity Status,Economic Count,Industry,Industry Count,Ethnic Group,Ethnic Count,Distance travelled to work,Travel Distance Count,Car Ownership,Car Ownership Count
0,0-4,34378,Female,17115,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,1 car or van in household,3139
1,0-4,34378,Female,17115,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,2 cars or vans in household,3034
2,0-4,34378,Female,17115,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,3 or more cars or vans in household,1723
3,0-4,34378,Female,17115,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,No cars or vans in household,1013
4,0-4,34378,Female,17115,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,30km to less than 60km,0,1 car or van in household,3139


(13099968, 16)


### 4. Zero-count filtering

Remove the rows with count = 0

In [4]:
COUNT_COLUMNS = [
    "Age Population",
    "Sex Population",
    "Household Count",
    "Economic Count",
    "Industry Count",
    "Ethnic Count",
    "Travel Distance Count",
    "Car Ownership Count",
]

profiles_before_zero_filter = len(profile_space)

positive_source_support = (
    profile_space[COUNT_COLUMNS]
    .notna()
    .all(axis=1)
    &
    profile_space[COUNT_COLUMNS]
    .gt(0)
    .all(axis=1)
)

profiles_removed_by_zero_filter = (
    ~positive_source_support
).sum()

profile_space = (
    profile_space.loc[positive_source_support]
    .copy()
    .reset_index(drop=True)
)

print(
    "Profiles before zero-count filtering:",
    f"{profiles_before_zero_filter:,}",
)
print(
    "Profiles removed:",
    f"{profiles_removed_by_zero_filter:,}",
)
print(
    "Profiles remaining:",
    f"{len(profile_space):,}",
)

Profiles before zero-count filtering: 13,099,968
Profiles removed: 5,437,648
Profiles remaining: 7,662,320


### 5. Validation

Economic activity is not applicable to profiles aged 0–4 or 5–11.

`Economic Activity Status = Does not apply`

In [5]:
CHILDREN = [
    "0-4",
    "5-11",
]

CHILDREN = profile_space[
    "Age"
].isin(CHILDREN)

age_rule_passed = (
    (CHILDREN == False)
    |
    (
        profile_space[
            "Economic Activity Status"
        ]
        == "Does not apply"
    )
)

profiles_before_age_rule = len(
    profile_space
)

profiles_removed_by_age_rule = (
    age_rule_passed == False
).sum()

profile_space = (
    profile_space.loc[
        age_rule_passed
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Profiles before age rule:",
    f"{profiles_before_age_rule:,}",
)

print(
    "Profiles removed by age rule:",
    f"{profiles_removed_by_age_rule:,}",
)

print(
    "Profiles remaining:",
    f"{len(profile_space):,}",
)

Profiles before age rule: 7,662,320
Profiles removed by age rule: 0
Profiles remaining: 7,662,320


### 6. Employment Rule

Industry and distance travelled to work are employment-related attributes.

Profiles classified as employed must have an applicable industry and work-distance category.

For profiles not classified as employed, both attributes must be recorded as `Does not apply`.

Profiles that do not satisfy these conditions are removed from the representative demographic population space.

In [7]:
EMPLOYED_PROFILES = [
    ("Economically active (excluding full-time students): In employment"),

    ("Economically active and a full-time student: In employment"),
]

DOES_NOT_APPLY = "Does not apply"

is_employed = profile_space[
    "Economic Activity Status"
].isin(EMPLOYED_PROFILES)

industry_applies = (
    profile_space["Industry"]
    != DOES_NOT_APPLY
)

work_distance_applies = (
    profile_space[
        "Distance travelled to work"
    ]
    != DOES_NOT_APPLY
)

employed_rule_passed = (
    is_employed
    & industry_applies
    & work_distance_applies
)

not_employed_rule_passed = (
    (is_employed == False)
    & (industry_applies == False)
    & (work_distance_applies == False)
)

employment_rule_passed = (
    employed_rule_passed
    | not_employed_rule_passed
)

profiles_before_employment_rule = len(
    profile_space
)

profiles_removed_by_employment_rule = (
    employment_rule_passed == False
).sum()

profile_space = (
    profile_space.loc[
        employment_rule_passed
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Profiles before employment rule:",
    f"{profiles_before_employment_rule:,}",
)

print(
    "Profiles removed by employment rule:",
    f"{profiles_removed_by_employment_rule:,}",
)

print(
    "Profiles remaining:",
    f"{len(profile_space):,}",
)

Profiles before employment rule: 7,662,320
Profiles removed by employment rule: 5,514,712
Profiles remaining: 2,147,608


In [8]:
is_employed = profile_space[
    "Economic Activity Status"
].isin(EMPLOYED_PROFILES)

employed_profiles = profile_space.loc[
    is_employed
]

not_employed_profiles = profile_space.loc[
    is_employed == False
]

employed_industry_valid = (
    employed_profiles["Industry"]
    != DOES_NOT_APPLY
).all()

employed_distance_valid = (
    employed_profiles[
        "Distance travelled to work"
    ]
    != DOES_NOT_APPLY
).all()

not_employed_industry_valid = (
    not_employed_profiles["Industry"]
    == DOES_NOT_APPLY
).all()

not_employed_distance_valid = (
    not_employed_profiles[
        "Distance travelled to work"
    ]
    == DOES_NOT_APPLY
).all()

assert employed_industry_valid
assert employed_distance_valid
assert not_employed_industry_valid
assert not_employed_distance_valid
assert profile_space[COUNT_COLUMNS].notna().all().all()
assert profile_space[COUNT_COLUMNS].gt(0).all().all()

print("Source-count validation: PASS")

print("Employment rule validation: PASS")

print(
    "Employed profiles:",
    f"{len(employed_profiles):,}",
)

print(
    "Not-employed profiles:",
    f"{len(not_employed_profiles):,}",
)

Source-count validation: PASS
Employment rule validation: PASS
Employed profiles: 2,110,976
Not-employed profiles: 36,632


### 7. Household Rules

Household categories must be compatible with the age of the representative profile.

Three rules are applied:

1. `Older family household` is restricted to the older age groups.
2. Profiles aged 0–4 or 5–11 cannot belong to households in which all children are non-dependent.
3. Profiles aged 0–4 or 5–11 cannot belong to a couple household with no children.

Profiles that do not satisfy these rules are removed.

In [9]:
OLDER_FRAMEWORK_AGE_GROUPS = [
    "65-74",
    "75+",
]

OLDER_FAMILY_HOUSEHOLD = (
    "Older family household: All members aged 66+"
)

NON_DEPENDENT_CHILDREN_HOUSEHOLDS = [
    (
        "Couple household: "
        "All children non-dependent"
    ),
    (
        "Lone parent household: "
        "All children non-dependent"
    ),
]

NO_CHILDREN_HOUSEHOLDS = [
    "Couple household: No children",
]

required_household_categories = {
    OLDER_FAMILY_HOUSEHOLD,
    *NON_DEPENDENT_CHILDREN_HOUSEHOLDS,
    *NO_CHILDREN_HOUSEHOLDS,
}

available_household_categories = set(
    profile_space[
        "Household Category"
    ].unique()
)

missing_household_categories = (
    required_household_categories
    - available_household_categories
)

if missing_household_categories:
    raise ValueError(
        "Missing household categories: "
        f"{sorted(missing_household_categories)}"
    )

print("Required household categories found.") 
# This allows that in the future, if we add more household categories, we don't have to update this code. We just need to ensure that the required categories are present.

Required household categories found.


In [10]:
# Apply household rules to the population pool
age = profile_space["Age"]

household_type = profile_space[
    "Household Category"
]

children = age.isin(
    CHILDREN
)

OLDER_FAMILY_HOUSEHOLD_rule_passed = (
    (
        household_type
        != OLDER_FAMILY_HOUSEHOLD
    )
    |
    age.isin(OLDER_FRAMEWORK_AGE_GROUPS)
)

non_dependent_children_rule_passed = (
    (children == False)
    |
    (
        household_type.isin(
            NON_DEPENDENT_CHILDREN_HOUSEHOLDS
        )
        == False
    )
)

no_children_household_rule_passed = (
    (children == False)
    |
    (
        household_type.isin(
            NO_CHILDREN_HOUSEHOLDS
        )
        == False
    )
)

household_rules_passed = (
    OLDER_FAMILY_HOUSEHOLD_rule_passed
    & non_dependent_children_rule_passed
    & no_children_household_rule_passed
)

profiles_before_household_rules = len(
    profile_space
)

population_removed_by_household_rules = (
    household_rules_passed == False
).sum()

profile_space = (
    profile_space.loc[
        household_rules_passed
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Population before household rules:",
    f"{profiles_before_household_rules:,}",
)

print(
    "Population removed by household rules:",
    f"{population_removed_by_household_rules:,}",
)

print(
    "Population remaining:",
    f"{len(profile_space):,}",
)

Population before household rules: 2,147,608
Population removed by household rules: 0
Population remaining: 2,147,608


In [11]:
older_family_household_profiles = (
    profile_space.loc[
        profile_space[
            "Household Category"
        ]
        == OLDER_FAMILY_HOUSEHOLD
    ]
)

children_profiles = (
    profile_space.loc[
        profile_space["Age"].isin(
            CHILDREN
        )
    ]
)

older_family_household_age_valid = (
    older_family_household_profiles[
        "Age"
    ]
    .isin(OLDER_FRAMEWORK_AGE_GROUPS)
    .all()
)

children_not_in_non_dependent_households = (
    children_profiles[
        "Household Category"
    ]
    .isin(
        NON_DEPENDENT_CHILDREN_HOUSEHOLDS
    )
    .any()
    == False
)

children_not_in_no_children_households = (
    children_profiles[
        "Household Category"
    ]
    .isin(
        NO_CHILDREN_HOUSEHOLDS
    )
    .any()
    == False
)

assert older_family_household_age_valid
assert (
    children_not_in_non_dependent_households
)
assert (
    children_not_in_no_children_households
)

print("Household rules validation: PASS")

Household rules validation: PASS


## 7. Final Validation

The final representative demographic population space is validated against all age, employment, and household rules.

The validation confirms that:

- young children have no applicable economic activity;
- employment-related attributes agree with employment status;
- Household Category is compatible with age; and
- no duplicate demographic populations remain.

In [12]:
ECONOMIC_INACTIVE_STATUSES = [
    "Economically inactive (excluding full-time students)",
    "Economically inactive and a full-time student",
]

economic_inactive_profiles = profile_space[
    profile_space[
        "Economic Activity Status"
    ].isin(ECONOMIC_INACTIVE_STATUSES)
]

inactive_with_employment_information = economic_inactive_profiles[
    economic_inactive_profiles["Industry"].ne("Does not apply")
    | economic_inactive_profiles[
        "Distance travelled to work"
    ].ne("Does not apply")
]

print(
    "Inactive populations:",
    f"{len(economic_inactive_profiles):,}",
)

print(
    "Inactive populations with employment information:",
    f"{len(inactive_with_employment_information):,}",
)

display(
    inactive_with_employment_information[
        [
            "Economic Activity Status",
            "Industry",
            "Distance travelled to work",
        ]
    ].value_counts().reset_index(name="Count")
)

Inactive populations: 17,632
Inactive populations with employment information: 0


,Economic Activity Status,Industry,Distance travelled to work,Count


In [13]:
CHILDREN = profile_space[
    "Age"
].isin(CHILDREN)

is_employed = profile_space[
    "Economic Activity Status"
].isin(EMPLOYED_PROFILES)

is_older_family_household = (
    profile_space["Household Category"]
    == OLDER_FAMILY_HOUSEHOLD
)

children_valid = (
    profile_space.loc[
        CHILDREN,
        "Economic Activity Status",
    ]
    == DOES_NOT_APPLY
).all()

employed_industry_valid = (
    profile_space.loc[
        is_employed,
        "Industry",
    ]
    != DOES_NOT_APPLY
).all()

employed_distance_valid = (
    profile_space.loc[
        is_employed,
        "Distance travelled to work",
    ]
    != DOES_NOT_APPLY
).all()

not_employed_industry_valid = (
    profile_space.loc[
        is_employed == False,
        "Industry",
    ]
    == DOES_NOT_APPLY
).all()

not_employed_distance_valid = (
    profile_space.loc[
        is_employed == False,
        "Distance travelled to work",
    ]
    == DOES_NOT_APPLY
).all()

older_family_household_valid = (
    profile_space.loc[
        is_older_family_household,
        "Age",
    ]
    .isin(OLDER_FRAMEWORK_AGE_GROUPS)
    .all()
)

children_household_valid = (
    profile_space.loc[
        CHILDREN,
        "Household Category",
    ]
    .isin(
        NON_DEPENDENT_CHILDREN_HOUSEHOLDS
        + NO_CHILDREN_HOUSEHOLDS
    )
    .any()
    == False
)

validation_results = {
    "Young children and economic activity": (
        children_valid
    ),
    "Employed profiles and industry": (
        employed_industry_valid
    ),
    "Employed profiles and work distance": (
        employed_distance_valid
    ),
    "Not-employed profiles and industry": (
        not_employed_industry_valid
    ),
    "Not-employed profiles and work distance": (
        not_employed_distance_valid
    ),
    "Older household and age": (
        older_family_household_valid
    ),
    "Young children and Household Category": (
        children_household_valid
    ),
}

validation_summary = pd.DataFrame(
    [
        {
            "Rule": rule,
            "Validation": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
        for rule, passed
        in validation_results.items()
    ]
)

display(validation_summary)

failed_rules = [
    rule
    for rule, passed
    in validation_results.items()
    if passed == False
]

if failed_rules:
    raise ValueError(
        "Final validation failed: "
        f"{failed_rules}"
    )

print("All representative population rules: PASS")

print(
    "Final population shape:",
    profile_space.shape,
)

,Rule,Validation
0,Young children and economic activity,PASS
1,Employed profiles and industry,PASS
2,Employed profiles and work distance,PASS
3,Not-employed profiles and industry,PASS
4,Not-employed profiles and work distance,PASS
5,Older household and age,PASS
6,Young children and Household Category,PASS


All representative population rules: PASS
Final population shape: (2147608, 16)


In [14]:
PROFILE_ATTRIBUTES = [
    "Age",
    "Sex",
    "Household Category",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

duplicate_populations = (
    profile_space
    .duplicated(
        subset=PROFILE_ATTRIBUTES
    )
    .sum()
)

complete_row_duplicates = (
    profile_space
    .duplicated()
    .sum()
)

duplicate_summary = pd.DataFrame(
    [
        {
            "Duplicate Type": (
                "Demographic population duplicates"
            ),
            "Count": duplicate_populations,
        },
        {
            "Duplicate Type": (
                "Complete row duplicates"
            ),
            "Count": complete_row_duplicates,
        },
    ]
)

display(duplicate_summary)

if duplicate_populations > 0:
    raise ValueError(
        "Duplicate demographic populations found: "
        f"{duplicate_populations:,}"
    )

if complete_row_duplicates > 0:
    raise ValueError(
        "Completely identical rows found: "
        f"{complete_row_duplicates:,}"
    )

print("Duplicate validation: PASS")

,Duplicate Type,Count
0,Demographic population duplicates,0
1,Complete row duplicates,0


Duplicate validation: PASS


## 8. Export

The validated representative profile space is exported for integration with the Behavioural Evidence Layer.

Each row represents one logically feasible demographic combination. It does not represent an individual synthetic person or a known joint population weight.

Supporting Census counts are retained for provenance and source validation, but they are not treated as individual behavioural attributes.

In [15]:
output_path.mkdir(
    parents=True,
    exist_ok=True,
)

constrained_profile_space = (
    output_path
    / (
        "constrained_representative_"
        "profile_space.parquet"
    )
)

profile_space.to_parquet(
    constrained_profile_space,
    index=False,
)

print(
    "Constrained representative profile space "
    "exported to:"
)

print(constrained_profile_space)

print(
    "Exported shape:",
    profile_space.shape,
)

Constrained representative profile space exported to:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\representative_profiles\constrained_representative_profile_space.parquet
Exported shape: (2147608, 16)
